# Section J — Short-answer questions (4 marks)
Answers to J1–J4, each ≤ 80 words. One mark per question.

In [0]:
# J1 – EXPLAIN FORMATTED: fact_order_line JOIN dim_product, sum net_amount by category_l1
print(spark.sql("""
EXPLAIN FORMATTED
SELECT p.category_l1, SUM(f.net_amount) AS total_net
FROM fk_ce_2026_07_1382.gold.fact_order_line f
JOIN fk_ce_2026_07_1382.gold.dim_product p ON f.product_id = p.product_id
GROUP BY p.category_l1
""").collect()[0][0])

## J1. Physical join operator and shuffle

**Physical join:** `PhotonBroadcastHashJoin Inner`

The engine chose broadcast hash join because `dim_product` is a small dimension table — the plan collects it to `SinglePartition` then broadcasts via `EXECUTOR_BROADCAST`, avoiding a costly shuffle of the large `fact_order_line`. The aggregation then shuffles by `category_l1` using `hashpartitioning(category_l1, 16)`.

**Shuffle operator in the plan:** `PhotonShuffleExchangeSink` (node 9)

## J2. Managed vs external table on DROP

For a **managed table**, `DROP TABLE` deletes both the metastore metadata and the underlying data files in cloud storage — the data is permanently lost. For an **external table**, `DROP TABLE` removes only the metastore metadata; the data files remain untouched in their external location and can be re-registered.

## J3. VACUUM and time travel

No. VACUUM with default settings deletes unreferenced files older than 7 days. Eight days after Run 2, the pre-Batch-2 files are unreferenced and exceed the 7-day threshold, so VACUUM removes them. The `VERSION AS OF` time-travel query fails because the data files for that version no longer exist. The controlling table property is `delta.deletedFileRetentionDuration` (default `interval 7 days`).

## J4. Promote Dev → Prod with DABs

Configuration file: `databricks.yml`. The `resources.jobs` section defines the job (tasks, clusters, schedule). A `variables` section defines a `catalog` variable, and the `targets` section defines per-environment overrides — e.g., `dev` sets `catalog: fk_dev`, `prod` sets `catalog: fk_prod`.